<a href="https://colab.research.google.com/github/vruddhis/semanticshift/blob/main/table2_disruption_drift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# TODO:TABLE 3 AND TABLE 5
# get files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
informal_embed_path = '/content/drive/MyDrive/semantic_shift/datasets/embeddings/informal_embeds.csv'
formal_embed_path = '/content/drive/MyDrive/semantic_shift/datasets/embeddings/formal_embeds.csv'

df_inf = pd.read_csv(informal_embed_path)
df_inf.head()

df_f= pd.read_csv(formal_embed_path)
df_f.head()

,word,event,register,period,sentence,timestamp,embedding
0,tag,social-media,formal,before,The service is backed by some well-lnown chara...,2008-06-18T17:49:07Z,"[-0.10919109731912613, 0.28668826818466187, 0...."
1,tag,social-media,formal,before,Here's the next part: if they need to fill in ...,2008-09-26T12:16:17Z,"[0.05975320562720299, 0.1855640560388565, -0.1..."
2,tag,social-media,formal,before,The £199 price tag didn't bother him - the Rea...,2008-09-04T09:23:00Z,"[-0.08915523439645767, 0.341594934463501, -0.1..."
3,tag,social-media,formal,before,"Or for those watching the pennies, there's the...",2008-11-18T00:01:00Z,"[0.09958921372890472, 0.007577353622764349, 0...."
4,tag,social-media,formal,before,The intro or article abstract Many publication...,2007-11-19T07:44:17Z,"[0.08138493448495865, 0.03273022547364235, 0.7..."


In [20]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from scipy.stats import entropy

In [46]:
def process_data(file_path, register_label):
    ALPHA = 0.5
    BETA = 0.5
    print(f"--- Processing {register_label} Register ---")

    # 1. Load data and IMMEDIATELY reset index to match the embedding stack
    df = pd.read_csv(file_path).reset_index(drop=True)

    def parse_emb(x):
        return np.fromstring(x.strip('[]'), sep=',')

    print("Parsing embeddings...")
    embeddings_raw = np.stack(df['embedding'].apply(parse_emb).values)

    def get_entropy(emb_subset, n_clusters=10):
        if len(emb_subset) < n_clusters: return 0
        mbk = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, n_init="auto")
        clusters = mbk.fit_predict(emb_subset)
        _, counts = np.unique(clusters, return_counts=True)
        return entropy(counts / counts.sum())

    results = []
    words = df['word'].unique()

    for word in words:
        # Get the word-specific data
        word_data = df[df['word'] == word]

        # Pull indices relative to the NEW 0-based index
        idx_before = word_data[word_data['period'] == 'before'].index
        idx_event  = word_data[word_data['period'] == 'event'].index
        idx_after  = word_data[word_data['period'] == 'after'].index

        # If we don't have enough data, skip
        if len(idx_before) < 10 or len(idx_after) < 10:
            continue

        # metadata
        event_id = word_data['event'].iloc[0] if 'event' in word_data.columns else "N/A"

        # Entropy calculations
        h_before = get_entropy(embeddings_raw[idx_before])
        h_after  = get_entropy(embeddings_raw[idx_after])

        # Disruption Score Logic
        before_embs = embeddings_raw[idx_before]
        chunks = np.array_split(before_embs, 5)
        h_baseline_samples = [get_entropy(c) for c in chunks if len(c) > 5]

        sigma_base = np.std(h_baseline_samples) if len(h_baseline_samples) > 1 else 0.01
        mu_base = np.mean(h_baseline_samples) if len(h_baseline_samples) > 0 else 0.1

        # FIX: If event index is empty, we print it so you know why scores are 0
        if len(idx_event) == 0:
            h_event = mu_base
        else:
            h_event = get_entropy(embeddings_raw[idx_event])

        # Scores
        drift = (h_after - h_before) / 3.32
        disruption = (sigma_base / (mu_base + 1e-9)) * 10
        h_final = (ALPHA * drift) + (BETA * disruption)

        base_info ={
            'target word': word,
            'event id': event_id,
            'register': register_label,
            'entropy shift': h_final,
            'drift score': drift,
            'disruption score': disruption
        }

        results.append(base_info)

        # # Append all 4 types
        # for s_type in ['baseline', 'disruption', 'short term drift', 'long term drift']:
        #     results.append({**base_info, 'shift type': s_type})

    final_meme_table = pd.DataFrame(results)

    # Return everything you need
    return final_meme_table[[
        'target word', 'event id', 'register', 'entropy shift',
         'drift score', 'disruption score'
    ]]

In [47]:
# Process both registers
informal_results = process_data(informal_embed_path, 'Informal')
formal_results = process_data(formal_embed_path, 'Formal')

# comparison_inf = informal_results[informal_results['shift type'] == 'long term drift']
# comparison_for = formal_results[formal_results['shift type'] == 'long term drift']

# # Merge them into one table for final analysis
# final_comparison = pd.merge(
#     comparison_inf,
#     comparison_for,
#     on='target word',
#     suffixes=('_inf', '_for')
# )

# Calculate Register Delta for both metrics
# final_comparison['drift_delta'] = abs(final_comparison['drift_score_inf'] - final_comparison['drift_score_for'])
# final_comparison['disruption_delta'] = abs(final_comparison['disruption_score_inf'] - final_comparison['disruption_score_for'])

# disruptive score
# display(final_comparison.sort_values(by='disruption_score_inf', ascending=False).head(5))

# drift score
# display(final_comparison.sort_values(by='drift_delta', ascending=False).head(5))

print("Disruption informal")
display(informal_results.sort_values(by='disruption score', ascending=False).head(5))

print("Disruption formal")
display(formal_results.sort_values(by='disruption score', ascending=False).head(5))

print("Drift informal")
display(informal_results.sort_values(by='drift score', ascending=False).head(5))

print("Drift formal")
display(formal_results.sort_values(by='drift score', ascending=False).head(5))

 # Save your final scores back to Drive so you don't lose them!
# final_comparison.to_csv('/content/drive/MyDrive/MEME_Project/final_meme_scores.csv', index=False)
# print("Success! Final scores saved to Drive.")

--- Processing Informal Register ---
Parsing embeddings...
--- Processing Formal Register ---
Parsing embeddings...
Disruption informal


,target word,event id,register,entropy shift,drift score,disruption score
1,binge,streaming,Informal,0.215808,0.022924,0.408692
5,algorithm,ai-automation,Informal,0.172582,-0.012242,0.357407
7,catfish,catfish-show,Informal,0.131421,-0.065383,0.328225
0,creator,social-media,Informal,0.133660,-0.028603,0.295922
6,bubble,covid-19,Informal,0.099762,0.002295,0.197230


Disruption formal


,target word,event id,register,entropy shift,drift score,disruption score
7,tweet,social-media,Formal,1.251885,0.118340,2.385430
8,remote,covid-19,Formal,0.500073,0.016116,0.984031
4,variant,covid-19,Formal,0.526248,0.083414,0.969082
11,bubble,covid-19,Formal,0.363194,-0.025010,0.751399
5,token,nft,Formal,0.376765,0.012477,0.741054


Drift informal


,target word,event id,register,entropy shift,drift score,disruption score
1,binge,streaming,Informal,0.215808,0.022924,0.408692
6,bubble,covid-19,Informal,0.099762,0.002295,0.197230
3,beta,fanfic,Informal,0.052824,-0.004908,0.110557
2,cancel,me-too,Informal,0.026892,-0.005126,0.058909
5,algorithm,ai-automation,Informal,0.172582,-0.012242,0.357407


Drift formal


,target word,event id,register,entropy shift,drift score,disruption score
7,tweet,social-media,Formal,1.251885,0.118340,2.385430
4,variant,covid-19,Formal,0.526248,0.083414,0.969082
0,tag,social-media,Formal,0.175887,0.057174,0.294601
10,stream,streaming,Formal,0.205392,0.019026,0.391758
8,remote,covid-19,Formal,0.500073,0.016116,0.984031
